In [8]:
import numpy as np
import torch
import torch.nn.functional as F

In [9]:
### Building a triagram

words = open('names.txt','r').read().splitlines()
chars = sorted(list(set("".join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}



In [10]:
xs, ys = [], []
for w in words:
  chs = ['.'] + list(w) + ['.']
  for ch1, ch2, ch3 in zip(chs, chs[1:], chs[2:]):
    ix1, ix2, ix3 = stoi[ch1], stoi[ch2], stoi[ch3]
    xs.append([ix1,ix2])
    ys.append(ix3)

xs, ys = torch.tensor(xs), torch.tensor(ys)


In [11]:
g = torch.Generator().manual_seed(2147483647)
W = torch.randn((27*27, 27), generator=g, requires_grad=True)

In [12]:
num, _ = xs.shape
xen = xs @ torch.tensor([27,1])
xenc = F.one_hot(xen, num_classes=27*27).float()
for k in range(30):
    logits = xenc @ W # predict log-counts
    counts = logits.exp() # counts, equivalent to N
    probs = counts / counts.sum(1, keepdims=True) # probabilities for next character
    loss = -probs[torch.arange(num), ys].log().mean()
    print(loss.item())
    W.grad = None # set to zero the gradient
    loss.backward()
    
    # update
    W.data += -100 * W.grad

3.72312331199646
3.574551820755005
3.445275068283081
3.3351171016693115
3.242729902267456
3.165238857269287
3.0992467403411865
3.0419602394104004
2.9914863109588623
2.9465696811676025
2.9063222408294678
2.8700695037841797
2.8372750282287598
2.8074951171875
2.780355930328369
2.7555322647094727
2.7327382564544678
2.711724281311035
2.692270517349243
2.6741890907287598
2.657317876815796
2.641519784927368
2.6266794204711914
2.6126978397369385
2.5994904041290283
2.5869853496551514
2.575119733810425
2.5638387203216553
2.5530953407287598
2.542847156524658


In [13]:
W

tensor([[ 1.5674, -0.2373, -0.0274,  ..., -0.0707,  2.4968,  2.4448],
        [-1.3904,  0.3132,  0.3786,  ..., -0.6929,  0.3005, -0.0728],
        [-0.2526,  1.0927, -0.1035,  ..., -0.8315, -0.2478, -1.2880],
        ...,
        [-0.5846,  0.4032, -0.3311,  ..., -0.5078, -2.0352, -0.1585],
        [ 0.8607, -0.6430, -1.2908,  ..., -1.0594,  0.4099, -1.0275],
        [ 0.1403,  0.2230,  0.5397,  ...,  1.2059, -0.0808,  1.3626]],
       requires_grad=True)

In [18]:
P = W.exp()
P /= P.sum(dim=1, keepdim=True)
for i in range(10):
    ix = 0
    p = torch.tensor([1/26 for _ in range(26)])
    n=0
    ix0 = torch.multinomial(p,num_samples=1, replacement=True, generator=g).item() + 1
    name = itos[ix0]
    while True:

        prob = P[ix*27 + ix0] #got rid of the one hot encoding
        ix1 = torch.multinomial(prob,num_samples=1, replacement=True, generator=g).item()
        if ix1 == 0:
            break
        ix, ix0 = ix0, ix1
        name = name + itos[ix1]

    print(name) #the model isn't suppose to be better as the data stills small, there's a lot more sparsity in W with trigram

fhcir
frena
cdwmkpdzsemiffna
ufynn
milnlkdnrsshospwfnqkeitrhyroqzhtqptthzqxxa
ann
vk
naymvrevbnqbtsywgbalen
htle
htltpcairsipeaunqgqexhzrhbycheyjickeah
